# Follow a copper ore through crushing and grinding

**Question:** How do requested grind targets change achieved size and grinding demand in one closed circuit?

Level: 1. Prerequisites: None; begin here. Reading time: 18 minutes.



## Objectives

Starting from one declared OreFlow ore and its nominal operating point, follow feed size through the comminution circuit. The lesson will distinguish a simulated flowsheet record from plant measurement and identify the quantities the engine actually computes.

## Setup

```bash
pip install numpy==2.2.6 scipy==1.17.1 pandas==2.2.3 matplotlib==3.11.2 nbclient==0.11.0 nbformat==5.11.1 ipykernel==7.4.0 SALib==1.6.0 scikit-learn==1.5.2 ruff==0.15.18 pytest==9.1.1
```

The OreFlow pipeline is imported from `data-pipeline/`; this notebook never installs packages.

For the level 1 01-comminution lesson, record the pinned versions before analysis so the environment behind its tables and figures can be reconstructed.

In [ ]:
# Import analysis tools and record their exact versions.
import importlib.metadata
import json
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

repo = Path("../../../").resolve()
sys.path.insert(0, str(repo / "data-pipeline"))
for package in ("numpy", "scipy", "pandas", "matplotlib", "nbclient", "nbformat", "ipykernel"):
    print(package, importlib.metadata.version(package))
legend_style = dict(loc="lower center", bbox_to_anchor=(0.5, 1.02), frameon=False)


## Data

Owner: OreFlow authors, MIT project licence. The production case catalog supplies the soft-porphyry ore, crusher, mill, classifier and nominal operating definition. Physical constants and the energy-law reference endpoints come from engine/data/constants.json. Circuit outputs and size distributions are calculated here, rather than loaded from a saved trace. No measured feed distribution, power campaign or new external dataset is introduced. Source: https://github.com/fsantibanezleal/CAOS_OreFlow.

## Method

The engine represents size on a fixed logarithmic grid. Crushing maps the feed distribution through the Whiten classification model. Grinding applies an energy-specific selection function and a breakage matrix, while the classifier sends each size class to overflow or recycle. At steady state, fresh feed equals final products within the unit balance tolerance. The lesson calls `simulate` from OreFlow's pipeline; no comminution equation is reimplemented here.

Import the OreFlow case catalog and circuit solver from the repository pipeline. The nominal point is the case's documented operating state; the solver returns the same structured result used by the bake.

In [ ]:
# Select one authored porphyry case and run OreFlow's production circuit solver.
from pipeline.cases.catalog import CASES
from pipeline.engine.circuit import simulate

case = next((item for item in CASES if item.id == "copper_porphyry_soft"))
result = simulate(case.ore, case.plant, case.nominal)
print("family:", case.plant.family)
print("throughput:", case.nominal.throughput_tph, "t/h")
print("feed F80:", round(result.metrics["crusher_feed_f80_um"], 1), "um")
print("product P80:", round(result.grinding.p80_um, 1), "um")
print("grinding energy:", round(result.grinding.specific_energy_kwh_t, 2), "kWh/t")


Inspect the solver's cumulative passing distributions at the named circuit boundaries. P80 is a percentile of a distribution, not a label that can be inferred from one representative particle.

In [ ]:
# Plot persisted grid curves returned by the same solved circuit.
plt.rcParams.update({"font.size": 18, "axes.titlepad": 40})
size = np.asarray(result.curves["size_um"])
psd = result.curves["psd"]
fig, ax = plt.subplots(figsize=(6.4, 3.8))
for stream in ("crusher_feed", "new_feed", "cyclone_overflow"):
    ax.plot(size, 100.0 * np.asarray(psd[stream]), label=stream.replace("_", " "))
ax.set_xscale("log")
ax.set_xlim(max(1.0, float(size.min())), float(size.max()))
ax.set(
    xlabel="Particle size (µm)",
    ylabel="Cumulative\npassing (%)",
    title="Nominal size distributions",
)
ax.grid(True, alpha=0.25)
ax.legend(["Crusher", "Fresh", "Overflow"], ncol=3, **legend_style)
fig.subplots_adjust(left=0.22, right=0.96, bottom=0.23, top=0.72)
ax.locator_params(axis="x", numticks=4)
ax.tick_params(axis="x", which="minor", labelbottom=False)


The cumulative arrays returned by the engine are fractions. Multiplication by 100 places every stream on the labelled percent scale, with 80% interpreted on the same basis. The displayed size domain spans the actual grid, including the crusher-feed F80 crossing rather than clipping it below the feed percentile. Separation of the three curves describes size reduction within one mass-balanced circuit; it does not establish measured ore calibration.

Read the computed stage metrics and maximum balance residual. A small closure error is a numerical conservation check for this model run; it does not validate the model against a plant.

In [ ]:
# Present operating outputs with explicit units and the solver's independent balance audit.
from IPython.display import display
import pandas as pd

metrics = result.metrics
rows = [
    ("Crusher feed F80", metrics["crusher_feed_f80_um"], "µm"),
    ("Crusher product P80", metrics["crusher_p80_um"], "µm"),
    ("Circuit product P80", metrics["p80_um"], "µm"),
    ("Specific grinding energy", metrics["specific_energy_grinding_kwh_t"], "kWh/t"),
    ("Maximum relative balance error", result.balance["max_relative_error"], "fraction"),
]
display(pd.DataFrame(rows, columns=["Quantity", "Value", "Unit"]))


Connect the size curves above to an actual mill-feed endpoint. Solve the soft porphyry again and take its achieved F80 directly from the grinding result. The crusher feed is a different stream, so substituting its coarser F80 would exaggerate the mill reduction. Keep the work index in kWh/t and the sizes in micrometres. These quantities let the analysis compare an energy requirement with a solved circuit without guessing which stream an endpoint represents.

In [ ]:
# Resolve the nominal circuit before evaluating alternative energy hypotheses.
from pipeline.cases.catalog import CASE_BY_ID
from pipeline.engine.circuit import simulate
from pipeline.engine.energy import law_curves

case = CASE_BY_ID["copper_porphyry_soft"]
nominal = simulate(case.ore, case.plant, case.nominal)
f80 = nominal.grinding.feed_f80_um
wi = case.nominal.work_index_kwh_t
print("Achieved mill F80 (µm):", round(f80, 2))


Evaluate Bond, Rittinger and Kick at the achieved nominal product. The returned columns are alternative estimates, each expressed in kWh per tonne. They are not three pieces of the mill's energy bill. Bond uses inverse square-root size, Rittinger inverse size, and Kick the logarithm of a size ratio. Their differences already matter at one point, even before a sweep reveals their shapes. Retain the values beside the independently computed grinding energy rather than selecting whichever happens to look closest.

In [ ]:
# Calculate the calibrated work at the achieved product size.
at_product = law_curves(wi, f80, np.array([nominal.grinding.p80_um]))
display(pd.DataFrame(at_product).round(3))


Now request a product twenty percent finer, keeping ore, water and fresh feed unchanged. The circuit must resolve its classifier recycle instead of simply replacing the printed P80. Read requested and achieved size separately: installed power can prevent the target being reached. The reported required power answers whether the request is physically achievable in this authored configuration. A size target is therefore an input to an interconnected calculation, and cannot be interpreted as a guaranteed output or an instruction to a real plant.

In [ ]:
# Ask the engine to achieve a finer product while holding the fresh feed and ore fixed.
fine_point = case.nominal.with_values(target_p80_um=case.nominal.target_p80_um * 0.8)
fine = simulate(case.ore, case.plant, fine_point)
print("Requested product P80 (µm):", fine_point.target_p80_um)
print("Achieved product P80 (µm):", round(fine.grinding.p80_um, 3))
print("Required power (kW):", round(fine.grinding.required_power_kw, 3))


## Results

**Answer:** Requesting <!-- result:Requested fine P80 (µm) -->0<!-- /result --> µm produces an achieved grind of <!-- result:Achieved fine P80 (µm) -->0<!-- /result --> µm, compared with nominal <!-- result:Nominal P80 (µm) -->0<!-- /result --> µm. The fine request requires <!-- result:Fine required power (kW) -->0<!-- /result --> kW. The solved recycle circuit, rather than the requested input alone, determines whether that product is reached.

Collect achieved nominal/fine grind and required power from the solved circuit for the grind-target answer.

In [ ]:
# Summarize the computed outcomes used in this lesson's interpretation.
answer_values = {
    "Nominal P80 (µm)": nominal.grinding.p80_um,
    "Requested fine P80 (µm)": fine_point.target_p80_um,
    "Achieved fine P80 (µm)": fine.grinding.p80_um,
    "Fine required power (kW)": fine.grinding.required_power_kw,
    "Bond work (kWh/t)": at_product["bond"][0],
    "Rittinger work (kWh/t)": at_product["rittinger"][0],
    "Kick work (kWh/t)": at_product["kick"][0],
}
answer_values = {label: round(float(number), 3) for label, number in answer_values.items()}
print(json.dumps(answer_values, ensure_ascii=False))


## Summary

The lesson executed the same circuit solver used by OreFlow, traced a size distribution and read its conservation check. Keep the distinction between model closure (internal consistency) and validation against measured plant data.

## References

- https://doi.org/10.1016/0301-7516(80)90034-4
- https://doi.org/10.1016/0301-7516(80)90034-4